# Customer Support Voice Assistant (Voice Bot) — Colab Project

**Pipeline:** Mic (browser) → Speech-to-Text (faster-whisper) → RAG retrieval (LangChain + ChromaDB) → LLM response (Groq llama-3.1-8b-instant) → Text-to-Speech (edge-tts) → Audio playback

This notebook is built to run end-to-end in **Google Colab**. Since Colab has no physical microphone access via `sounddevice`, we use the **browser's microphone through JavaScript** to record audio, then feed it into the pipeline.

**Companion file:** `support_docs.txt` — upload this in Step 4 to build the knowledge base (RAG) that grounds the bot's answers in actual support content.

---
### Steps in this notebook
1. Install dependencies
2. Set up Groq API key
3. Speech-to-Text (STT) with faster-whisper
4. Build the RAG knowledge base from `support_docs.txt`
5. LLM response generation (grounded, voice-friendly)
6. Text-to-Speech (TTS) with edge-tts
7. Full voice bot orchestration loop
8. Test run


## Step 1 — Install dependencies


In [ ]:
!pip install -q groq langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma \
    faster-whisper soundfile numpy edge-tts pydub nest_asyncio

print("✅ Dependencies installed.")

## Step 2 — Set up your Groq API key

Get a free key from https://console.groq.com/keys

Uses Colab's secret manager (`userdata`). Add a secret named `GROQ` (Colab left sidebar → 🔑 icon) containing your key from https://console.groq.com/keys, and make sure "Notebook access" is toggled on for it.

In [ ]:
# Access your secret keys in Python
from google.colab import userdata
import os

## Store as env variable
groq_key = userdata.get('GROQ')

os.environ["GROQ_API_KEY"] = groq_key
print("✅ API key set.")


## Step 3 — Speech-to-Text (STT)

### 3a. Record audio using the browser microphone

Colab notebooks run on a remote server, so `sounddevice` cannot access your local mic directly.
Instead, we use a small JavaScript snippet to record audio in your **browser** and send it back to Python.

In [ ]:
from IPython.display import Javascript
from google.colab import output
from base64 import b64decode

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))

var recorder, gumStream, chunks;

async function record(sec) {
  chunks = [];
  gumStream = await navigator.mediaDevices.getUserMedia({audio: true});
  recorder = new MediaRecorder(gumStream);

  recorder.ondataavailable = e => chunks.push(e.data);   // attached BEFORE start()

  const stopped = new Promise(resolve => {
    recorder.onstop = resolve;   // resolves only after stop() fully completes
  });

  recorder.start();
  await sleep(sec * 1000);
  recorder.stop();
  await stopped;

  gumStream.getTracks().forEach(track => track.stop());

  const blob = new Blob(chunks, { type: "audio/webm" });
  const arrayBuffer = await blob.arrayBuffer();
  return btoa(String.fromCharCode(...new Uint8Array(arrayBuffer)));
}
"""

def record_audio_browser(duration=5, filename="input_audio.webm"):
    print(f"🎙️ Recording for {duration} seconds... speak now!")
    output.eval_js(RECORD_JS)
    b64_audio = output.eval_js(f"record({duration})")
    audio_bytes = b64decode(b64_audio)
    with open(filename, "wb") as f:
        f.write(audio_bytes)
    print("✅ Recording saved:", filename)
    return filename

### 3b. Transcribe audio with faster-whisper

`faster-whisper` uses ffmpeg internally (already installed in Colab), so it can read the `.webm` file directly — no manual conversion needed.

Changes for speed + reliability:
- `"small"` model (better accuracy than `"base"` without a huge speed hit)
- `beam_size=1` — greedy decoding instead of beam search; noticeably faster with only a small accuracy cost for short utterances
- `vad_filter=True` — skips silent stretches instead of wasting time transcribing them
- Returns `(text, avg_logprob)` so low-confidence transcriptions can be caught before they reach the LLM

In [ ]:
from faster_whisper import WhisperModel

# bumped from "base" to "small" for better accuracy
stt_model = WhisperModel("small", device="cpu", compute_type="int8")

def transcribe(audio_path):
    segments, info = stt_model.transcribe(
        audio_path,
        language="en",
        beam_size=1,            # faster: greedy decoding
        vad_filter=True,        # skip silence instead of transcribing it
        vad_parameters=dict(min_silence_duration_ms=300)
    )
    segments = list(segments)  # materialize generator so it can be reused safely

    text = " ".join(seg.text for seg in segments).strip()

    if segments:
        avg_logprob = sum(seg.avg_logprob for seg in segments) / len(segments)
    else:
        avg_logprob = -999  # no segments detected at all -> treat as very low confidence

    return text, avg_logprob

print("✅ STT model loaded.")

### 3c. Quick test — record and transcribe

In [ ]:
audio_path = record_audio_browser(duration=5)
transcribed_text, confidence = transcribe(audio_path)
print("You said:", transcribed_text)
print("Confidence (avg_logprob):", round(confidence,2))

## Step 4 — Build the RAG knowledge base

Upload the companion file **`support_docs.txt`** (provided separately) using the file picker below.
This file contains sample customer-support content (FAQs, policies, troubleshooting) which the bot will retrieve from to ground its answers — this is the "RAG" part of the pipeline.

You can replace `support_docs.txt` with your own company's real support content later; the format (plain text, one topic per section) stays the same.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from google.colab import files

print("📤 Please upload support_docs.txt")
uploaded = files.upload()  # select support_docs.txt when prompted

In [ ]:
# !pip install -q -U langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

def build_vectorstore(filepath="support_docs.txt", persist_dir="chroma_db"):
    with open(filepath, "r", encoding="utf-8") as f:
        raw_text = f.read()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", " "]
    )
    chunks = splitter.split_text(raw_text)
    documents = [Document(page_content=chunk) for chunk in chunks]

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    vectordb = Chroma.from_documents(
        documents,
        embeddings,
        collection_metadata={"hnsw:space": "cosine"},
        persist_directory=persist_dir
    )
    print(f"✅ Vectorstore built with {len(documents)} chunks.")
    return vectordb

vectordb = build_vectorstore("support_docs.txt")

In [ ]:
# Quick sanity check — test retrieval
test_query = "Can i cancle my order?"
results = vectordb.similarity_search(test_query, k=2)
for i, r in enumerate(results, 1):
    print(f"--- Match {i} ---")
    print(r.page_content)
    print()

## Step 5 — LLM response generation (grounded + voice-friendly)

Voice responses need to be **short and conversational** (no markdown, no bullet points) since they'll be spoken aloud by the TTS engine.

In [ ]:
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

SYSTEM_PROMPT = """You are a helpful customer support voice assistant.
Rules:
- Keep answers short (2-3 sentences max), since they will be spoken aloud.
- Never use bullet points, markdown, numbers lists, or symbols.
- Only answer using the provided context from support documents.
- If the knowledge base doesn't contain the answer, politely say you'll connect them to a human agent.
- Be warm, clear, and efficient, like a good support representative.
"""

def get_response(user_query, chat_history=None, k=2):
    docs = vectordb.similarity_search(user_query, k=k)
    context = "\n".join(d.page_content for d in docs)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if chat_history:
        messages.extend(chat_history)

    messages.append({
        "role": "user",
        "content": f"Context from support docs:\n{context}\n\nCustomer question: {user_query}"
    })

    completion = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        temperature=0.3,
        max_tokens=120
    )
    return completion.choices[0].message.content.strip()

print("✅ Response engine ready.")

In [ ]:
# Quick test
test_reply = get_response("What is your refund policy?")
print("Bot:", test_reply)

## Step 6 — Text-to-Speech (TTS)

We use `edge-tts` — free, natural-sounding voices, no API key required.
Colab can play audio directly in the notebook output using `IPython.display.Audio`.

- `nest_asyncio.apply()` so `asyncio.run()` works inside Colab/Jupyter's already-running event loop
- `speak()` now waits for the **actual playback duration** (measured from the generated mp3 with `pydub`) before returning, so the next recording never starts while the bot is still talking

In [ ]:
import asyncio
import nest_asyncio
import edge_tts
from IPython.display import Audio, display
from pydub import AudioSegment
import time

nest_asyncio.apply()

VOICE = "en-US-JennyNeural"

async def _synthesize(text, path):
    communicate = edge_tts.Communicate(text, VOICE)
    await communicate.save(path)

def speak(text, filename="bot_reply.mp3", autoplay=True):
    asyncio.run(_synthesize(text, filename))
    duration_sec = len(AudioSegment.from_file(filename)) / 1000.0  # ms -> sec
    if autoplay:
        display(Audio(filename, autoplay=True))
    time.sleep(duration_sec + 0.8)   # wait for playback to finish + small buffer
    return filename

In [ ]:

# Quick test
speak("Hello! Thanks for reaching out. How can I help you today?")

## Step 7 — Full voice bot orchestration loop

This ties everything together:
`record (browser mic) → transcribe → retrieve + generate grounded reply → speak reply`

The loop keeps context (`chat_history`) across turns and exits when the caller says a closing phrase.

1. **Direct human requests end the call immediately** — if the user says something like "connect me to a human" / "talk to an agent" on their own (not just in response to a bot offer), the bot hands off and ends the call in that same turn, no confirmation round-trip needed.
2. **Escalation offered by the bot** still asks for a yes/no on the next turn, but a "yes" now ends the call right there (this was already working from your last version — kept as is).
3. **Low-confidence / junk transcriptions** are rejected before hitting the LLM, so garbled audio doesn't produce a confusing answer.
4. Recording window defaults to **4 seconds** to keep turns snappy; combined with `beam_size=1` + VAD filtering + capped chat history + `max_tokens=120`, each turn should feel noticeably faster end-to-end.

In [ ]:
EXIT_PHRASES = ["bye", "goodbye", "that's all", "end call", "thank you bye"]

# phrases that mean "I want a human" when the USER says them directly
DIRECT_HUMAN_REQUEST = [
    "human agent", "talk to a human", "speak to a human", "connect me to a human",
    "connect me to human", "real person", "human assistance", "speak with an agent",
    "talk to an agent", "connect to a human", "connect with a human"
]

# used when the BOT offered escalation and we're waiting on a yes/no
ESCALATION_OFFER_TRIGGERS = ["connect you to a human", "human agent", "connect you with a human"]
CONFIRM_WORDS = ["yes", "yeah", "sure", "please", "ok", "okay", "go ahead"]
DECLINE_WORDS = ["no", "nah", "not now", "never mind"]

CONFIDENCE_THRESHOLD = -1.0   # tune based on your own test runs (closer to 0 = stricter)

def is_direct_human_request(user_text: str) -> bool:
    return any(phrase in user_text.lower() for phrase in DIRECT_HUMAN_REQUEST)

def is_escalation_offer(bot_reply: str) -> bool:
    return any(trigger in bot_reply.lower() for trigger in ESCALATION_OFFER_TRIGGERS)

def simulate_human_handoff():
    ticket_id = f"TCK-{int(time.time()) % 100000}"
    msg = f"Connecting you to a human agent now. Your ticket reference is {ticket_id}. An agent will follow up within 24 hours. Goodbye!"
    print(f"🔄 {msg}")
    speak(msg)
    return ticket_id


def run_voice_bot(turn_duration=4, max_turns=10):
    print("🤖 Voice support bot started.\n")
    chat_history = []
    pending_escalation = False

    for turn in range(max_turns):
        print(f"\n===== Turn {turn + 1} =====")
        audio_path = record_audio_browser(duration=turn_duration)
        user_text, confidence = transcribe(audio_path)

        # reject empty / no real words / low-confidence transcriptions
        if not user_text.strip() or not any(c.isalnum() for c in user_text):
            print("⚠️ Didn't catch that, please try again.")
            continue

        if confidence < CONFIDENCE_THRESHOLD:
            print(f"⚠️ Low confidence ({confidence:.2f}) — asking user to repeat.")
            msg = "Sorry, I didn't quite catch that. Could you please repeat?"
            print("Bot:", msg)
            speak(msg)
            continue

        print(f"You: {user_text}")

        # -------- exit phrases --------
        if any(phrase in user_text.lower() for phrase in EXIT_PHRASES):
            closing = "Thanks for calling. Have a great day!"
            print("Bot:", closing)
            speak(closing)
            break

        # -------- user directly asks for a human, unprompted --------
        if is_direct_human_request(user_text):
            simulate_human_handoff()
            break

        # -------- user is responding to a pending escalation offer --------
        if pending_escalation:
            if any(word in user_text.lower() for word in CONFIRM_WORDS):
                simulate_human_handoff()
                break
            elif any(word in user_text.lower() for word in DECLINE_WORDS):
                pending_escalation = False
                reply = "No problem, I'm still here to help. What else can I do for you?"
                print("Bot:", reply)
                speak(reply)
                chat_history.append({"role": "user", "content": user_text})
                chat_history.append({"role": "assistant", "content": reply})
                continue
            # if unclear, fall through to normal handling below

        # -------- normal RAG-grounded response --------
        reply = get_response(user_text, chat_history)
        print(f"Bot: {reply}")

        chat_history.append({"role": "user", "content": user_text})
        chat_history.append({"role": "assistant", "content": reply})

        speak(reply)

        pending_escalation = is_escalation_offer(reply)

    print("\n✅ Call ended.")

print("✅ Orchestration ready.")

## Step 8 — Run the voice bot

Run the cell below. On each turn, allow microphone access in your browser when prompted, then speak your question within the recording window.
Say **"bye"**, **"goodbye"**, or **"that's all"** to end the call.

**Questions to test that map directly to support_docs.txt**

**(say these clearly, close to the mic, in a quiet room):**

Try these — short, clear, and matched closely to actual section headers, so Whisper has an easier time and retrieval will be clean:

"What is your refund policy?"

"How do I return an item?"

"How can I track my order?"

"My order is delayed, what should I do?"

"Can I cancel my order?"

"My payment failed, what happens now?"

"I can't log in to my account, what should I do?"

"When will this item be back in stock?"

"Does this come with a warranty?"

"Can I speak to a human agent?"

In [ ]:
run_voice_bot(turn_duration=5, max_turns=6)